In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import hstack

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
dev_df = pd.read_csv("../preprocessing/clean_data_with_clean_target.csv")


required_cols = ["prev_clean", "target_clean", "label_id", "group"]

assert all(c in dev_df.columns for c in required_cols)
# assert all(c in test_df.columns for c in ["prev_clean", "target_clean", "label_id"])

dev_df = dev_df.reset_index(drop=True)
# test_df = test_df.reset_index(drop=True)

print("Development dataset:", dev_df.shape)
# print("Separate test dataset:", test_df.shape)
print("Development classes:", sorted(dev_df["label_id"].unique()))

Development dataset: (19431, 10)
Development classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [4]:
splitter = StratifiedGroupKFold(
  n_splits=10,
  shuffle=True,
  random_state=42
)

dev_df["fold"] = -1

for fold_number, (_, val_indices) in enumerate(
  splitter.split(
    X=dev_df,
    y=dev_df["label_id"],
    groups=dev_df["group"]
  )
):
  dev_df.loc[val_indices, "fold"] = fold_number

print(dev_df["fold"].value_counts().sort_index())

fold
0    1887
1    2111
2    1970
3    2000
4    1851
5    1862
6    1987
7    1953
8    1883
9    1927
Name: count, dtype: int64


In [5]:
def build_features(train_df, val_df):
  # Extract context and target text
  train_target = train_df["target_clean"].fillna("").astype(str)
  train_prev = train_df["prev_clean"].fillna("").astype(str)

  val_target = val_df["target_clean"].fillna("").astype(str)
  val_prev = val_df["prev_clean"].fillna("").astype(str)

  # Target word features
  target_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Target character features
  target_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Context word features
  prev_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
  )

  # Fit vectorizers only on the current training fold
  X_train_tw = target_word.fit_transform(train_target)
  X_train_tc = target_char.fit_transform(train_target)
  X_train_pw = prev_word.fit_transform(train_prev)

  # Transform validation fold using training vocabularies
  X_val_tw = target_word.transform(val_target)
  X_val_tc = target_char.transform(val_target)
  X_val_pw = prev_word.transform(val_prev)

  # Join features in the same order
  X_train = hstack(
    [X_train_tw, X_train_tc, X_train_pw],
    format="csr"
  )

  X_val = hstack(
    [X_val_tw, X_val_tc, X_val_pw],
    format="csr"
  )

  return X_train, X_val

In [6]:
fold_results = []

for fold_number in range(10):
  print(f"\n========== Fold {fold_number} ==========")

  train_fold = dev_df[dev_df["fold"] != fold_number]
  val_fold = dev_df[dev_df["fold"] == fold_number]

  # Separate labels
  y_train = train_fold["label_id"]
  y_val = val_fold["label_id"]

  # Build features for this fold
  X_train, X_val = build_features(train_fold, val_fold)

  # Train Logistic Regression
  model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=42
  )

  model.fit(X_train, y_train)

  # Validate
  val_pred = model.predict(X_val)

  macro_f1 = f1_score(
    y_val, val_pred, average="macro", zero_division=0
  )

  weighted_f1 = f1_score(
    y_val, val_pred, average="weighted", zero_division=0
  )

  accuracy = accuracy_score(y_val, val_pred)

  fold_results.append({
    "fold": fold_number,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "accuracy": accuracy
  })

  print("Training rows:", len(train_fold))
  print("Validation rows:", len(val_fold))
  print(f"Macro F1:    {macro_f1:.4f}")
  print(f"Weighted F1: {weighted_f1:.4f}")
  print(f"Accuracy:    {accuracy:.4f}")

  # Release fold-specific objects
  del X_train, X_val, model


========== Fold 0 ==========
Training rows: 17544
Validation rows: 1887
Macro F1:    0.9647
Weighted F1: 0.9773
Accuracy:    0.9772

========== Fold 1 ==========
Training rows: 17320
Validation rows: 2111
Macro F1:    0.9517
Weighted F1: 0.9712
Accuracy:    0.9711

========== Fold 2 ==========
Training rows: 17461
Validation rows: 1970
Macro F1:    0.9757
Weighted F1: 0.9853
Accuracy:    0.9853

========== Fold 3 ==========
Training rows: 17431
Validation rows: 2000
Macro F1:    0.9763
Weighted F1: 0.9821
Accuracy:    0.9820

========== Fold 4 ==========
Training rows: 17580
Validation rows: 1851
Macro F1:    0.9656
Weighted F1: 0.9765
Accuracy:    0.9762

========== Fold 5 ==========
Training rows: 17569
Validation rows: 1862
Macro F1:    0.9737
Weighted F1: 0.9808
Accuracy:    0.9807

========== Fold 6 ==========
Training rows: 17444
Validation rows: 1987
Macro F1:    0.9579
Weighted F1: 0.9707
Accuracy:    0.9703

========== Fold 7 ==========
Training rows: 17478
Validation rows: 1

In [7]:
results_df = pd.DataFrame(fold_results)

display(results_df)

print("\nCross-validation summary:")
print(
  results_df[["macro_f1", "weighted_f1", "accuracy"]]
  .agg(["mean", "std"])
)

,fold,macro_f1,weighted_f1,accuracy
0,0,0.964712,0.977331,0.977213
1,1,0.951669,0.971157,0.971104
2,2,0.975747,0.985337,0.985279
3,3,0.976252,0.982136,0.982000
4,4,0.965627,0.976456,0.976229
5,5,0.973684,0.980800,0.980666
6,6,0.957936,0.970688,0.970307
7,7,0.963931,0.972738,0.972862
8,8,0.970502,0.977921,0.977695
9,9,0.960559,0.978000,0.977686



Cross-validation summary:
      macro_f1  weighted_f1  accuracy
mean  0.966062     0.977256  0.977104
std   0.008065     0.004762  0.004774


In [8]:
# Prepare text
X_dev_target = dev_df["target_clean"].fillna("").astype(str)
X_dev_prev = dev_df["prev_clean"].fillna("").astype(str)

test_df = pd.read_csv("../preprocessing/clean_test_data_with_preprocessing.csv")  # Change to your actual filename

X_test_target = test_df["reply_clean"].fillna("").astype(str)
X_test_prev = test_df["context_clean"].fillna("").astype(str)

# Fit the three vectorizers on all development data
tfidf_target_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_target_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_prev_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
)

X_dev_tw = tfidf_target_word.fit_transform(X_dev_target)
X_dev_tc = tfidf_target_char.fit_transform(X_dev_target)
X_dev_pw = tfidf_prev_word.fit_transform(X_dev_prev)

# Transform the separate test set
X_test_tw = tfidf_target_word.transform(X_test_target)
X_test_tc = tfidf_target_char.transform(X_test_target)
X_test_pw = tfidf_prev_word.transform(X_test_prev)

# Combine features
X_dev = hstack([X_dev_tw, X_dev_tc, X_dev_pw], format="csr")
X_test = hstack([X_test_tw, X_test_tc, X_test_pw], format="csr")

# Train final model
final_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

final_model.fit(X_dev, dev_df["label_id"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [9]:
y_test = test_df["label_id"]
test_pred = final_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, test_pred))
print(
  "Test macro F1:",
  f1_score(y_test, test_pred, average="macro", zero_division=0)
)
print(
  "Test weighted F1:",
  f1_score(y_test, test_pred, average="weighted", zero_division=0)
)

print("\nClassification report:")
print(classification_report(y_test, test_pred, zero_division=0))

Test accuracy: 0.3276053215077605
Test macro F1: 0.13102259332707025
Test weighted F1: 0.2962248511033836

Classification report:
              precision    recall  f1-score   support

           0       0.46      0.48      0.47       786
           1       0.16      0.35      0.22        20
           2       0.33      0.01      0.02        88
           3       0.09      0.19      0.12        26
           4       0.25      0.08      0.12        13
           5       0.00      0.00      0.00        21
           6       0.32      0.53      0.40       346
           7       0.18      0.01      0.02       375
           8       0.12      0.09      0.10        57
           9       0.00      0.00      0.00        55
          10       0.03      0.14      0.05         7
          11       0.02      0.40      0.04        10

    accuracy                           0.33      1804
   macro avg       0.16      0.19      0.13      1804
weighted avg       0.32      0.33      0.30      1804

